<a href="https://colab.research.google.com/github/kouzi5816/ML_Engineering_Repo/blob/main/02_RAGApp/Docker_Notebook_with_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dockerファイルの作成

In [14]:
dockerfile = """
# ベースイメージ（Python 3.11 スリム版）
FROM python:3.11-slim

# 作業ディレクトリを設定
WORKDIR /app

# 依存関係を先にコピー（キャッシュを効かせるため）
COPY requirements.txt .

# パッケージをインストール
RUN pip install --no-cache-dir -r requirements.txt

# アプリのコードをコピー
COPY app.py .

# Cloud Runはポート8080を使う（Streamlitのデフォルトは8501なので変更）
EXPOSE 8080

# 起動コマンド
CMD streamlit run app.py --server.port=8080 --server.address=0.0.0.0 --server.headless=true
"""

with open("Dockerfile", "w", newline="\n") as f:
    f.write(dockerfile.strip())

print("Dockerfile を作成しました")

Dockerfile を作成しました


.dockerignoreファイルの作成（不要ファイルをイメージに含めない）

In [4]:
dockerignore = """
__pycache__
*.pyc
*.pyo
.env
.git
chroma_db
"""

with open(".dockerignore", "w", newline="\n") as f:
    f.write(dockerignore.strip())

print(".dockerignore を作成しました")

.dockerignore を作成しました


gcloud CLIのインストール

In [5]:
!curl https://sdk.cloud.google.com | bash -s -- --disable-prompts --install-dir=/usr/local
import os
os.environ["PATH"] += ":/usr/local/google-cloud-sdk/bin"

ストリーミング出力は最後の 5000 行に切り捨てられました。
google-cloud-sdk/lib/third_party/botocore/data/braket/2019-09-01/paginators-1.json
google-cloud-sdk/lib/third_party/botocore/data/braket/2019-09-01/service-2.json
google-cloud-sdk/lib/third_party/botocore/data/budgets/2016-10-20/endpoint-rule-set-1.json
google-cloud-sdk/lib/third_party/botocore/data/budgets/2016-10-20/paginators-1.json
google-cloud-sdk/lib/third_party/botocore/data/budgets/2016-10-20/service-2.json
google-cloud-sdk/lib/third_party/botocore/data/ce/2017-10-25/endpoint-rule-set-1.json
google-cloud-sdk/lib/third_party/botocore/data/ce/2017-10-25/paginators-1.json
google-cloud-sdk/lib/third_party/botocore/data/ce/2017-10-25/service-2.json
google-cloud-sdk/lib/third_party/botocore/data/chime-sdk-identity/2021-04-20/endpoint-rule-set-1.json
google-cloud-sdk/lib/third_party/botocore/data/chime-sdk-identity/2021-04-20/paginators-1.json
google-cloud-sdk/lib/third_party/botocore/data/chime-sdk-identity/2021-04-20/service-2.json
google-cloud-sdk/li

GCPの認証

In [6]:
!gcloud auth login --no-launch-browser

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=PtHiDtCa9Yns0tJUwD8aXane7drSgl&prompt=consent&token_usage=remote&access_type=offline&code_challenge=5BRwDr3ruN-HHg8Q9e9bgp3rbd5ZM-wsy40stkqXSyA&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AdkVLPyShv4Yx3oCCyDLuE0QAB-qkASKkHXeR2gpb_42Rt6kgYQjXRiE5wxsZP1cWOMmzA

You are now logged in as [kouzi5816@gmail.com].
Your current project is

プロジェクト情報の設定

In [7]:
PROJECT_ID = "rag-app-project-first-deploy"  # ← 自分のIDに変更
!gcloud config set project {PROJECT_ID}

Updated property [core/project].


必要なAPIの有効化

In [8]:
!gcloud services enable run.googleapis.com
!gcloud services enable artifactregistry.googleapis.com
!gcloud services enable cloudbuild.googleapis.com

Operation "operations/acf.p2-724625460069-bce5bd4e-a19c-4d4c-81e2-5711ec585016" finished successfully.
Operation "operations/acf.p2-724625460069-4168989d-4f8c-40bf-bdb2-a5b7bcb38f98" finished successfully.


Artifact Registryにリポジトリを作成（初回のみ）

In [9]:
REGION = "asia-northeast1"  # 東京リージョン
REPO   = "rag-app-repo"
IMAGE  = f"asia-northeast1-docker.pkg.dev/{PROJECT_ID}/{REPO}/rag-app"

!gcloud artifacts repositories create {REPO} \
    --repository-format=docker \
    --location={REGION}

Create request issued for: [rag-app-repo]
Created repository [rag-app-repo].


Docker認証の設定

In [10]:
!gcloud auth configure-docker asia-northeast1-docker.pkg.dev --quiet

`docker` and `docker-credential-gcloud` need to be in the same PATH in order to work correctly together.
gcloud's Docker credential helper can be configured but it will not work until this is corrected.
Adding credentials for: asia-northeast1-docker.pkg.dev
Docker configuration file updated.


Cloud Buildでイメージをビルド＆push（Colabにはdockerが入っていないため）

In [19]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

!gcloud builds submit \
    --tag {IMAGE} \
    /content/drive/MyDrive/RAG  # ← app.py等があるフォルダのパス

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating temporary archive of 7 file(s) totalling 23.9 MiB before compression.
Uploading tarball of [/content/drive/MyDrive/RAG] to [gs://rag-app-project-first-deploy_cloudbuild/source/1783396594.313073-3d4445d0e5da490699fa61fc6561a664.tgz]
Created [https://cloudbuild.googleapis.com/v1/projects/rag-app-project-first-deploy/locations/global/builds/e3ff1c66-5d0a-4a7f-aaa8-a76f4a87a6b2].
Logs are available at [ https://console.cloud.google.com/cloud-build/builds/e3ff1c66-5d0a-4a7f-aaa8-a76f4a87a6b2?project=724625460069 ].
Waiting for build to complete. Polling interval: 1 second(s).
 REMOTE BUILD OUTPUT
starting build "e3ff1c66-5d0a-4a7f-aaa8-a76f4a87a6b2"

FETCHSOURCE
Fetching storage object: gs://rag-app-project-first-deploy_cloudbuild/source/1783396594.313073-3d4445d0e5da490699fa61fc6561a664.tgz#1783396596945224
Copying gs://rag-app-project-first-deploy_cloud

実行結果の確認

In [16]:
!gcloud artifacts docker images list \
    asia-northeast1-docker.pkg.dev/{PROJECT_ID}/rag-app-repo

Listing items under project rag-app-project-first-deploy, location asia-northeast1, repository rag-app-repo.

IMAGE                                                                             DIGEST                                                                   CREATE_TIME          UPDATE_TIME          SIZE
asia-northeast1-docker.pkg.dev/rag-app-project-first-deploy/rag-app-repo/rag-app  sha256:c8a58ab661916fec57858fa3d3519199396aff3a0668807c23ae18e59f145e5f  2026-07-07T02:26:48  2026-07-07T02:26:48  307607856


アプリ用URLの発行

In [17]:
# APIキーの取得
from google.colab import userdata
# Google ColabのSecretsに保存したAPIキーを読み込み
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [20]:
SERVICE_NAME   = "rag-app"

!gcloud run deploy {SERVICE_NAME} \
    --image {IMAGE} \
    --platform managed \
    --region {REGION} \
    --allow-unauthenticated \
    --set-env-vars GEMINI_API_KEY={GEMINI_API_KEY} \
    --memory 1Gi \
    --port 8080

Deploying container to Cloud Run service [rag-app] in project [rag-app-project-first-deploy] region [asia-northeast1]
Service [rag-app] revision [rag-app-00002-646] has been deployed and is serving 100 percent of traffic.
Service URL: https://rag-app-724625460069.asia-northeast1.run.app
